# Batch analysis for multiple tickers

Edit `TICKERS` below and run the code cell. Each ticker gets its own folder containing the executed notebooks, HTML plots, and company description.

The `GEMINI_API_KEY` environment variable must be configured before running the batch.

In [11]:
import getpass
import os

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass(
        "Enter GEMINI_API_KEY (input hidden): "
    )
if not os.environ["GEMINI_API_KEY"].strip():
    raise RuntimeError("A non-empty GEMINI_API_KEY is required")

In [12]:
import json
import os
import re
import subprocess
import sys
from html import escape
from pathlib import Path

import nbformat
from nbconvert import HTMLExporter

# Add or remove ticker symbols here.
TICKERS = ["PG", "AAPL"]

ROOT = Path.cwd()
NOTEBOOKS = {
    "price_extractor": ROOT / "price_extractor.ipynb",
    "enterprise": ROOT / "enterprise.ipynb",
}
LOGO_URLS = {
    "PG": "https://upload.wikimedia.org/wikipedia/commons/8/85/Procter_%26_Gamble_logo.svg",
    "AAPL": "https://www.apple.com/favicon.ico",
}

for notebook_path in NOTEBOOKS.values():
    if not notebook_path.exists():
        raise FileNotFoundError(f"Missing notebook: {notebook_path}")


def run_notebook(notebook_path, ticker, output_dir):
    executed_name = f"{notebook_path.stem}_executed.ipynb"
    environment = os.environ.copy()
    environment["STOCK_TICKER"] = ticker
    command = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        str(notebook_path),
        "--output",
        executed_name,
        "--output-dir",
        str(output_dir),
        "--ExecutePreprocessor.timeout=600",
    ]
    result = subprocess.run(
        command,
        cwd=ROOT,
        env=environment,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        details = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(
            f"Failed to execute {notebook_path.name} for {ticker} "
            f"(exit code {result.returncode}).\n{details}"
        )
    return output_dir / executed_name


def save_description(executed_path, output_dir):
    notebook = json.loads(executed_path.read_text(encoding="utf-8"))
    text_parts = []
    for cell in notebook["cells"]:
        for output in cell.get("outputs", []):
            if output.get("output_type") == "stream" and output.get("name") == "stderr":
                continue
            text = output.get("text")
            if text:
                text_parts.extend(text if isinstance(text, list) else [text])
    description = "".join(text_parts).strip()
    (output_dir / "company_description.txt").write_text(
        description + "\n", encoding="utf-8"
    )


def render_notebook(executed_path):
    notebook = nbformat.read(executed_path, as_version=4)
    for cell in notebook.cells:
        cell["outputs"] = [
            output
            for output in cell.get("outputs", [])
            if not (output.get("output_type") == "stream" and output.get("name") == "stderr")
        ]
        for output in cell.get("outputs", []):
            plotly_data = output.get("data", {}).get("application/vnd.plotly.v1+json")
            if plotly_data:
                figure = json.dumps(plotly_data, separators=(",", ":"))
                output["data"]["text/html"] = (
                    '<div class="plotly-figure"></div>'
                    f"<script>Plotly.newPlot(document.currentScript.previousElementSibling, "
                    f"{figure}.data, {figure}.layout, {figure}.config || {{responsive:true}});</script>"
                )
    exporter = HTMLExporter()
    exporter.exclude_input = True
    rendered, _ = exporter.from_notebook_node(notebook)
    body_match = re.search(r"<body[^>]*>(.*?)</body>", rendered, re.IGNORECASE | re.DOTALL)
    if not body_match:
        raise RuntimeError(f"Could not render notebook body: {executed_path}")
    return body_match.group(1)


def build_report(ticker, enterprise_path, price_path, output_dir):
    enterprise_body = render_notebook(enterprise_path)
    price_body = render_notebook(price_path)
    logo_url = LOGO_URLS.get(ticker)
    logo = f'<img class="masthead-logo" src="{escape(logo_url)}" alt="{escape(ticker)} company logo">' if logo_url else ""
    report = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{escape(ticker)} | Stock prices analysis</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
:root {{ color-scheme: dark; --ink: #f4f1ea; --muted: #a7aaa8; --background: #111415; --panel: #1a1e1f; --line: #384041; --accent: #d6f36a; }}
* {{ box-sizing: border-box; }} body {{ margin: 0; background: #111415; color: var(--ink); font-family: Georgia, "Times New Roman", serif; }}
.report {{ max-width: 1480px; margin: 0 auto; padding: 42px 28px 72px; }}
.masthead {{ display: flex; justify-content: space-between; align-items: center; gap: 24px; padding-bottom: 28px; border-bottom: 1px solid var(--line); }}
.masthead-brand {{ display: flex; align-items: center; gap: 18px; }} .masthead-logo {{ width: 72px; height: 72px; object-fit: contain; background: #fff; border-radius: 50%; padding: 12px; }}
.eyebrow, h2, .ticker {{ font-family: Arial, sans-serif; }} .eyebrow {{ margin: 0 0 10px; color: var(--accent); font-size: 12px; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; }}
h1 {{ margin: 0; font-size: clamp(42px, 7vw, 82px); line-height: .92; font-weight: 400; }} .ticker {{ color: var(--accent); font-size: 16px; font-weight: 700; letter-spacing: 3px; }}
.overview {{ margin-top: 24px; }} .panel, .chart-output {{ background: var(--panel); border: 1px solid var(--line); border-radius: 8px; }} .panel {{ padding: 24px; }}
h2 {{ margin: 0 0 18px; color: var(--muted); font-size: 13px; letter-spacing: 1.6px; text-transform: uppercase; }} .enterprise-output pre {{ white-space: pre-wrap; font: 18px/1.55 Georgia, serif; color: var(--ink); }}
.charts {{ margin-top: 34px; }} .chart-output {{ padding: 8px; overflow: hidden; }} .plotly-figure {{ min-height: 360px; }}
@media (max-width: 760px) {{ .report {{ padding: 26px 14px 44px; }} .masthead {{ align-items: flex-start; }} .masthead-logo {{ width: 56px; height: 56px; }} .masthead-brand {{ gap: 12px; }} .panel {{ padding: 17px; }} }}
</style>
</head>
<body><main class="report">
<header class="masthead"><div class="masthead-brand">{logo}<div><p class="eyebrow">Stock prices analysis</p><h1>{escape(ticker)}</h1></div></div><div class="ticker">{escape(ticker)}</div></header>
<section class="overview"><article class="panel enterprise-output"><h2>Company profile</h2>{enterprise_body}</article></section>
<section class="charts"><h2>Market analysis</h2><div class="chart-output">{price_body}</div></section>
</main></body></html>"""
    report_path = output_dir / f"{ticker}_report.html"
    report_path.write_text(report, encoding="utf-8")
    return report_path


for raw_ticker in TICKERS:
    ticker = raw_ticker.strip().upper()
    invalid_characters = set('/\\:*?"<>|')
    if not ticker or any(character in invalid_characters for character in ticker):
        raise ValueError(f"Invalid ticker: {raw_ticker!r}")
    ticker_dir = ROOT / ticker
    ticker_dir.mkdir(exist_ok=True)
    print(f"Running {ticker}...")
    executed_notebooks = {}
    for notebook_name, notebook in NOTEBOOKS.items():
        executed_notebooks[notebook_name] = run_notebook(notebook, ticker, ticker_dir)
    save_description(executed_notebooks["enterprise"], ticker_dir)
    report_path = build_report(
        ticker,
        executed_notebooks["enterprise"],
        executed_notebooks["price_extractor"],
        ticker_dir,
    )
    print(f"Finished {ticker}: {report_path}")

print("Batch complete.")


Running PG...
Finished PG: d:\Materiały\git\stock-indices\PG\PG_report.html
Running AAPL...
Finished AAPL: d:\Materiały\git\stock-indices\AAPL\AAPL_report.html
Batch complete.
